# Persistent System Data Engineer Interview Experience

**Position:** Data Engineer
**Experience:** 4+ Years


## 🟡 Round 1 – Technical 1

**SQL | PySpark | Python**

1. Write a SQL query to find the **third-highest salary** from an employee table.
2. Explain the difference between **INNER JOIN, LEFT JOIN, RIGHT JOIN, and FULL JOIN** with use cases.
3. How do you handle **duplicate records in a PySpark DataFrame**?
4. Write a **PySpark transformation** to filter and aggregate large datasets.
5. What are **Lazy Evaluation and Actions in Spark**?
6. How do you **optimize a slow-running Spark job**?



## 1.

### Employee Table

| employee_id | employee_name | department | salary |
| ----------: | ------------- | ---------- | -----: |
|         101 | Amit          | IT         |  80000 |
|         102 | Rahul         | HR         |  60000 |
|         103 | Priya         | IT         |  95000 |
|         104 | Neha          | Finance    |  75000 |
|         105 | Rohit         | IT         |  90000 |
|         106 | Sneha         | HR         |  85000 |



### Approach 1: Using `DENSE_RANK()` — Recommended

```sql
SELECT employee_id,
       employee_name,
       salary
FROM (
    SELECT employee_id,
           employee_name,
           salary,
           DENSE_RANK() OVER (ORDER BY salary DESC) AS salary_rank
    FROM employee
) t
WHERE salary_rank = 3;
```

**Output:**

| employee_id | employee_name | salary |
| ----------: | ------------- | -----: |
|         106 | Sneha         |  85000 |

### Approach 2: Using `DISTINCT` + `LIMIT/OFFSET`

For databases supporting `LIMIT`:

```sql
SELECT DISTINCT salary
FROM employee
ORDER BY salary DESC
LIMIT 1 OFFSET 2;
```

**Output:**

```text
85000
```

### ⭐ Interview Point

Use **`DENSE_RANK()`** when you want the **3rd distinct-highest salary** and need to return **all employees** having that salary.

For example, if two employees earn ₹85,000, `DENSE_RANK()` returns both employees.


## 2.

**“The main difference between INNER, LEFT, RIGHT, and FULL JOIN is which unmatched records are retained.”**

Assume we have:

**Employee**

| emp_id | name  | dept_id |
| ------ | ----- | ------- |
| 1      | Amit  | 10      |
| 2      | Rahul | 20      |
| 3      | Priya | 30      |
| 4      | Neha  | 40      |

**Department**

| dept_id | dept_name |
| ------- | --------- |
| 10      | IT        |
| 20      | HR        |
| 30      | Finance   |
| 50      | Marketing |

### 1. INNER JOIN

Returns **only records that have a match in both tables**.

```sql
SELECT e.name, d.dept_name
FROM employee e
INNER JOIN department d
    ON e.dept_id = d.dept_id;
```

**Use case:** When I need only valid/matching records, such as employees who have a corresponding department.

---

### 2. LEFT JOIN

Returns **all records from the left table** and matching records from the right table. Unmatched right-side values become `NULL`.

```sql
SELECT e.name, d.dept_name
FROM employee e
LEFT JOIN department d
    ON e.dept_id = d.dept_id;
```

**Use case:** When I need all employees, including employees whose department information is missing.

---

### 3. RIGHT JOIN

Returns **all records from the right table** and matching records from the left table.

```sql
SELECT e.name, d.dept_name
FROM employee e
RIGHT JOIN department d
    ON e.dept_id = d.dept_id;
```

**Use case:** When the department table is the primary table and I want all departments, including departments with no employees.

---

### 4. FULL OUTER JOIN

Returns **all records from both tables**. Matching records are combined, while unmatched records from either side contain `NULL`.

```sql
SELECT e.name, d.dept_name
FROM employee e
FULL OUTER JOIN department d
    ON e.dept_id = d.dept_id;
```

**Use case:** Very useful for **source-target reconciliation**, where I need to identify records existing only in the source, only in the target, or in both.

### ⭐ Strong Interview Summary

> **INNER JOIN → matching records only**
> **LEFT JOIN → everything from left + matches from right**
> **RIGHT JOIN → everything from right + matches from left**
> **FULL JOIN → everything from both tables**

**Real-world Data Engineering example:**
For an ETL reconciliation, I would typically use a **FULL OUTER JOIN** to identify missing records on either side and validate source-vs-target data completeness.


## 3.

### Handling Duplicate Records in PySpark

**“In PySpark, the approach depends on whether I want to remove exact duplicates or duplicates based on specific business keys.”**

### 1. Remove exact duplicate rows

If the complete row is duplicated:

```python
df_clean = df.dropDuplicates()
```

Example:

```text
+---+-----+------+
|id |name |salary|
+---+-----+------+
|1  |Amit |50000 |
|2  |Rahul|60000 |
|1  |Amit |50000 |  <- duplicate
+---+-----+------+
```

```python
df_clean = df.dropDuplicates()
```

---

### 2. Remove duplicates based on specific columns

If `employee_id` is the business key:

```python
df_clean = df.dropDuplicates(["employee_id"])
```

This keeps one record for each `employee_id`.

---

### 3. Keep the latest record — Real-world scenario

If multiple records exist for the same employee and I need to retain the **latest record**, I use a window function.

```python
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window_spec = Window.partitionBy("employee_id") \
                    .orderBy(col("updated_at").desc())

df_clean = (
    df.withColumn("rn", row_number().over(window_spec))
      .filter(col("rn") == 1)
      .drop("rn")
)
```

### Example

| employee_id | name  | salary | updated_at |
| ----------- | ----- | -----: | ---------- |
| 101         | Amit  |  50000 | 2026-09-10 |
| 101         | Amit  |  55000 | 2026-09-15 |
| 102         | Rahul |  60000 | 2026-09-12 |

After deduplication:

| employee_id | name  | salary | updated_at |
| ----------- | ----- | -----: | ---------- |
| 101         | Amit  |  55000 | 2026-09-15 |
| 102         | Rahul |  60000 | 2026-09-12 |

### ⭐ Strong Interview Answer

> **“First, I identify the business key and understand what qualifies as a duplicate. For exact duplicates, I use `dropDuplicates()`. If duplicates are based on specific keys, I use `dropDuplicates(["key"])`. If I need deterministic selection, such as keeping the latest record, I use `row_number()` with a window partitioned by the business key and ordered by the timestamp descending. In production, I also validate duplicate counts before and after the transformation and consider data quality and performance implications.”**


## 4. 

### PySpark Filter & Aggregate

**Scenario:** We have a large `sales` DataFrame and need to calculate the **total sales and number of orders per customer** for completed orders above ₹1,000.

### Sample Data

| order_id | customer_id | amount | status    |
| -------- | ----------- | -----: | --------- |
| 101      | C01         |   1500 | Completed |
| 102      | C02         |    800 | Completed |
| 103      | C01         |   2500 | Completed |
| 104      | C03         |   3000 | Cancelled |
| 105      | C02         |   2000 | Completed |

### PySpark Code

```python
from pyspark.sql.functions import col, sum, count

result_df = (
    sales_df
    .filter(
        (col("status") == "Completed") &
        (col("amount") > 1000)
    )
    .groupBy("customer_id")
    .agg(
        sum("amount").alias("total_sales"),
        count("order_id").alias("order_count")
    )
)
```

### Output

| customer_id | total_sales | order_count |
| ----------- | ----------: | ----------: |
| C01         |        4000 |           2 |
| C02         |        2000 |           1 |

### ⭐ Interview-Level Explanation

> **“First, I apply filtering as early as possible to reduce the volume of data before the expensive aggregation. Then I group the filtered data by the required business key and use aggregation functions such as `sum`, `count`, or `avg`. For large datasets, I would also ensure that filters are pushed down to the source where possible, select only required columns, and monitor shuffle during the `groupBy` operation.”**

### Performance Considerations for Large Data

* **Filter early** → reduce data before shuffle.
* **Select required columns only** → reduce memory and network I/O.
* **Predicate pushdown** → let the source filter data when supported.
* **Avoid unnecessary `groupBy`** → aggregation causes shuffle.
* **Handle data skew** → especially when some customers have disproportionately large data.
* **Use appropriate partitioning** → particularly for frequently grouped/joined keys.
* For Delta/Parquet sources, leverage **partition pruning** where applicable.


## 5.

### Lazy Evaluation vs Actions in Spark

**“Spark follows lazy evaluation, meaning transformations are not executed immediately. Spark builds a logical execution plan and executes it only when an action is triggered.”**

### 1. Lazy Evaluation

**Transformations** are operations that create a new DataFrame/RDD but don't immediately execute the computation.

Examples:

```python
df_filtered = df.filter(df.salary > 50000)
df_selected = df_filtered.select("name", "salary")
```

At this point, Spark **hasn't actually processed the data**. It builds the execution plan.

Common transformations:

* `filter()`
* `select()`
* `withColumn()`
* `groupBy()`
* `join()`
* `dropDuplicates()`

### 2. Actions

An **action triggers Spark's execution** and produces a result or writes data.

Examples:

```python
df_selected.show()
```

Other common actions:

```python
df.count()
df.collect()
df.first()
df.write.parquet("/output/path")
```

When `show()` or another action is called, Spark executes the required transformations.

### Example

```python
df_filtered = df.filter(df.salary > 50000)   # Transformation
df_result = df_filtered.select("name", "salary")  # Transformation

df_result.show()  # Action → Execution starts
```

### Why Lazy Evaluation is Important

Spark uses lazy evaluation to **optimize the execution plan** before running it.

For example:

```python
df.filter(col("salary") > 50000) \
  .select("name", "salary") \
  .groupBy("name") \
  .count()
```

Instead of executing every operation separately, Spark can optimize the overall plan using **Catalyst Optimizer** and generate an efficient execution plan.

### ⭐ Strong Interview Answer

> **“Lazy evaluation means Spark doesn't execute transformations immediately. It records them as a lineage/execution plan and waits until an action is called. Actions such as `show()`, `count()`, `collect()`, and `write()` trigger the actual execution. This allows Spark's Catalyst Optimizer and execution engine to optimize the complete plan, reduce unnecessary computation, and improve performance.”**

**Easy way to remember:**
**Transformation → Builds the plan**
**Action → Executes the plan**


## 6.

### How to Optimize a Slow Spark Job

**“I first identify the bottleneck using Spark UI, then optimize based on whether the issue is related to data scanning, shuffle, joins, skew, memory, or resource configuration.”**

### 1. Check Spark UI First

I analyze:

* **Stages and tasks**
* Shuffle read/write
* Task execution time
* Data skew
* Spill to disk
* Executor memory/GC
* Number of partitions

### 2. Reduce Data Early

Apply filters and select only required columns as early as possible:

```python
df = (
    spark.read.parquet("/data")
    .filter(col("year") == 2026)
    .select("customer_id", "amount")
)
```

This reduces **I/O, memory usage, and shuffle volume**.

### 3. Optimize Joins

For a small lookup table, use **Broadcast Join**:

```python
from pyspark.sql.functions import broadcast

result = large_df.join(
    broadcast(small_df),
    "customer_id"
)
```

This can avoid a large shuffle.

### 4. Handle Data Skew

If one key contains a disproportionately large amount of data, I investigate **skewed partitions**.

Possible solutions:

* Salting
* Broadcast join where appropriate
* Adaptive Query Execution (AQE)
* Repartitioning

### 5. Optimize Partitions

Avoid both **too few and too many partitions**.

```python
df = df.repartition("customer_id")
```

For reducing partitions after filtering:

```python
df = df.coalesce(20)
```

I choose the partition count based on data size and cluster resources rather than using an arbitrary number.

### 6. Avoid Unnecessary Operations

For example, avoid:

```python
df.collect()
```

on large datasets because it brings data to the driver.

Also avoid unnecessary:

* `distinct()`
* `groupBy()`
* `repartition()`
* repeated actions

### 7. Cache Only When Needed

If the same DataFrame is reused multiple times:

```python
df.cache()
df.count()   # materialize cache
```

Otherwise, unnecessary caching can consume executor memory.

### 8. Optimize File Format & Storage

Prefer **Parquet/Delta** over raw CSV where appropriate and use:

* Partition pruning
* Predicate pushdown
* Appropriate file sizes
* Delta optimization techniques where applicable

### ⭐ Strong Interview Answer

> **“For a slow Spark job, I first use Spark UI to identify the bottleneck. Then I reduce input data through predicate pushdown and column pruning, optimize joins using broadcast joins when appropriate, handle data skew, tune partitioning, minimize expensive shuffles, and avoid unnecessary actions and caching. I also optimize the storage format and leverage AQE where appropriate. Finally, I compare execution time, shuffle volume, task distribution, and resource utilization before and after the changes.”**

**Key keywords to mention in an interview:**
**Spark UI → Shuffle → Partitioning → Broadcast Join → Data Skew → AQE → Predicate Pushdown → Caching → File Optimization → Resource Tuning**.


## 🔵 Round 2 – Technical 2

**ADF | Azure | Databricks**

1. Explain how you built an **ETL pipeline using Azure Data Factory**.
2. How do you implement **incremental loads using a watermark or timestamp**?
3. What is the **Azure Databricks architecture**?
4. What is the difference between **cache() and persist() in Spark**?
5. How do you handle **schema drift in ADF pipelines**?
6. How do you securely manage **secrets and credentials in Azure**?




## 1.

### Building an ETL Pipeline Using Azure Data Factory

> **“I typically design the ADF pipeline in multiple stages—ingestion, transformation, validation, and loading. For example, I can ingest data from an on-premise SQL Server into ADLS Gen2, transform it using Azure Databricks, and load the curated data into a target Azure SQL/Delta Lake layer.”**

### 1. Source & Connection

* Identify source: SQL Server, Oracle, REST API, SFTP, etc.
* Create **Linked Services** in ADF.
* For on-premises sources, configure **Self-hosted Integration Runtime**.
* Parameterize connections where possible.

### 2. Ingestion – Source → ADLS

Use **Copy Activity**:

```text
Source
  ↓
ADF Copy Activity
  ↓
ADLS Gen2
  ↓
Bronze/Raw Layer
```

For example:

```text
SQL Server → ADF → ADLS Gen2 /raw/customer/
```

I generally preserve the raw data before transformation for **reprocessing, auditing, and troubleshooting**.

### 3. Transformation

For complex transformations, I use **Azure Databricks**.

```text
ADLS Bronze
     ↓
Azure Databricks
     ↓
Silver / Curated
```

Typical transformations include:

* Data cleansing
* Type casting
* Deduplication
* Joins
* Business-rule implementation
* Null handling
* Aggregations

ADF can orchestrate the Databricks notebook using a **Databricks Notebook Activity**.

### 4. Load to Target

After transformation:

```text
Databricks
    ↓
ADLS Silver / Delta
    ↓
ADF
    ↓
Azure SQL / Synapse / Gold Layer
```

I use the appropriate sink depending on the downstream requirement.

### 5. Incremental Load

Instead of loading the complete dataset every time, I implement **incremental loading** using:

* Watermark column
* Last modified timestamp
* CDC
* Source-specific change tracking

Example:

```sql
SELECT *
FROM customer
WHERE modified_date > @last_watermark
```

The watermark can be maintained in a **control/metadata table**.

### 6. Error Handling & Monitoring

I configure:

* Success/Failure dependencies
* Retry policies
* Timeout settings
* Logging
* Azure Monitor/ADF monitoring
* Alerts for pipeline failures

For failed records, I can redirect them to an **error/quarantine layer** for investigation.

### 7. Parameterization

I avoid hardcoding values such as:

```text
Source path
Database name
Table name
File name
Environment
```

Instead, I use **pipeline/dataset parameters** so the same pipeline can be reused across Dev, Test, and Production.

### ⭐ Strong Interview Answer

> **“In my ADF ETL pipeline, I first configure Linked Services and Integration Runtime for the source and target systems. I use Copy Activity to ingest data into ADLS Gen2, usually maintaining a raw Bronze layer. For complex transformations, I orchestrate Databricks notebooks from ADF and create curated Silver/Gold data. I implement incremental loading using watermark/CDC logic instead of full loads. I also add parameterization, retries, failure handling, logging, monitoring, and alerts. Finally, I validate source-to-target counts and key business metrics before considering the pipeline successful.”**

### Simple Architecture

```text
Source Systems
     │
     ▼
Azure Data Factory
     │
     │ Copy Activity
     ▼
ADLS Gen2 - Bronze
     │
     ▼
Azure Databricks
     │
     │ Transform / Clean / Deduplicate
     ▼
ADLS - Silver/Gold
     │
     ▼
Azure SQL / Synapse / BI
```

**Interview keywords:**
`ADF → Linked Service → Integration Runtime → Copy Activity → ADLS Gen2 → Databricks → Incremental Load → Parameterization → Error Handling → Monitoring → Validation`


## 2.
### Incremental Load Using Watermark/Timestamp in ADF

> **“Instead of extracting the entire source table every time, I maintain a watermark representing the last successfully processed value. During each run, I extract only records whose `last_modified` value is greater than the previous watermark.”**

### Example

Suppose the source table is:

| customer_id | name  | amount | last_modified    |
| ----------- | ----- | -----: | ---------------- |
| 101         | Amit  |   5000 | 2026-09-18 10:00 |
| 102         | Rahul |   7000 | 2026-09-19 11:00 |
| 103         | Priya |   9000 | 2026-09-20 14:00 |
| 104         | Neha  |   6000 | 2026-09-21 10:00 |

Assume the **last successful watermark is**:

```text
2026-09-20 14:00
```

The next pipeline extracts:

```sql
SELECT *
FROM customer
WHERE last_modified > '2026-09-20 14:00';
```

So only **customer 104** is processed.

---

### ADF Pipeline Flow

```text
Control / Watermark Table
          ↓
Get Previous Watermark
          ↓
     Lookup Activity
          ↓
     Copy Activity
          ↓
Source WHERE last_modified > watermark
          ↓
     ADLS / Databricks
          ↓
Transformation & Load
          ↓
Get New Maximum Timestamp
          ↓
Update Watermark
```

### Implementation Steps

**1. Maintain a control table**

For example:

| pipeline_name | table_name | last_watermark   |
| ------------- | ---------- | ---------------- |
| Customer_Load | customer   | 2026-09-20 14:00 |

**2. Read the previous watermark**

Use an **ADF Lookup Activity** to retrieve the last successful timestamp.

**3. Pass it to the source query**

```sql
SELECT *
FROM customer
WHERE last_modified > @watermark;
```

In ADF, the query can be parameterized using pipeline/lookup values.

**4. Process the incremental data**

Load the extracted records into ADLS/Databricks/target database and perform the required transformations.

**5. Calculate the new watermark**

After successful processing:

```sql
SELECT MAX(last_modified)
FROM customer;
```

**6. Update the control table**

Store the new maximum timestamp as the watermark **only after the pipeline succeeds**.

---

### ⭐ Important Production Consideration

I would avoid updating the watermark before the target load succeeds.

```text
Extract → Transform → Load → Validate → Update Watermark
                                      ↑
                              Only after success
```

This prevents records from being skipped if the pipeline fails midway.

### Handling Same-Timestamp Records

A timestamp alone can cause problems if multiple records have exactly the same timestamp. For high-reliability pipelines, I can use a **composite watermark**:

```text
(last_modified_timestamp, primary_key)
```

or use **CDC/change tracking** where supported.

### Strong Interview Answer

> **“I implement incremental loading using a control table that stores the last successfully processed watermark. At the beginning of the pipeline, ADF reads this watermark using a Lookup activity and passes it to the source query. The query extracts only records where `last_modified > previous_watermark`. After successful transformation, target loading, and validation, I calculate the new maximum timestamp and update the control table. I update the watermark only after successful completion to prevent data loss. For cases where multiple records have the same timestamp, I use a composite watermark or CDC to make the process reliable.”**

**Key terms:** `Watermark → Lookup → Parameterized Query → Incremental Extraction → Control Table → Validation → Update Watermark → CDC`


## 3.

### Azure Databricks Architecture

> **“Azure Databricks follows a control plane and data plane architecture. The control plane manages the Databricks workspace and services, while the data plane is where the actual Spark compute runs and processes data.”**

### High-Level Architecture

```text
                Azure Databricks
                       │
          ┌────────────┴────────────┐
          │                         │
     Control Plane              Data Plane
          │                         │
  ┌───────┴────────┐         ┌──────┴─────────┐
  │ Workspace      │         │ Compute        │
  │ Notebooks      │         │ Clusters       │
  │ Jobs           │         │ Spark Workers  │
  │ Workflows      │         │ Spark Driver   │
  │ Security       │         └──────┬─────────┘
  └────────────────┘                │
                                    ▼
                              Azure Storage
                              ADLS Gen2
                              Delta Lake
```

### 1. Control Plane

The **control plane** is managed by Databricks.

It contains/manages:

* Workspace
* Notebooks
* Jobs & Workflows
* Cluster configuration
* Users and permissions
* Repos
* Job scheduling
* Workspace metadata

The control plane primarily handles **management and orchestration**, rather than processing the user's data.

### 2. Data Plane

The **data plane** contains the actual compute resources.

For example:

```text
Spark Driver
     │
 ┌───┼────────┐
 ▼   ▼        ▼
Worker Worker Worker
```

The Spark cluster performs:

* Data reading
* Transformations
* Joins
* Aggregations
* Machine learning
* Streaming processing

### 3. Storage Layer

Databricks typically works with Azure storage such as:

* **ADLS Gen2**
* Delta Lake
* Azure Blob Storage

For example:

```text
ADLS Gen2
   ↓
Bronze → Silver → Gold
   ↓
Delta Tables
```

### 4. Security & Integration

In an enterprise environment, Databricks integrates with Azure services such as:

* **Microsoft Entra ID** for identity
* **Azure Key Vault** for secrets
* **Unity Catalog** for governance, access control, and lineage
* **Azure Monitor/Log Analytics** for monitoring

### ⭐ Strong Interview Answer

> **“Azure Databricks has two major architectural components: the control plane and the data plane. The control plane manages the Databricks workspace, notebooks, jobs, workflows, and configurations. The data plane contains the Spark driver and worker nodes where data processing actually happens. Databricks can read and write data from ADLS Gen2 and use Delta Lake for reliable lakehouse storage. In an enterprise setup, I would typically integrate it with Entra ID for authentication, Unity Catalog for governance and access control, and Key Vault for secret management.”**

### Real-World Data Engineering Flow

```text
Source
  ↓
ADF / Kafka / APIs
  ↓
ADLS Gen2 - Bronze
  ↓
Azure Databricks
  ↓
Spark Transformations
  ↓
Delta Lake - Silver
  ↓
Business Aggregations
  ↓
Delta Lake - Gold
  ↓
Power BI / Synapse / Applications
```

**Interview keywords:**
**Control Plane → Data Plane → Spark Driver → Workers → ADLS Gen2 → Delta Lake → Unity Catalog → Entra ID → Key Vault → ADF**


## 4.
### `cache()` vs `persist()` in Spark

> **“Both `cache()` and `persist()` are used to store a DataFrame/RDD in memory or other storage so that Spark doesn't recompute it every time. The main difference is that `cache()` uses Spark's default storage level, while `persist()` allows me to explicitly choose the storage level.”**

### 1. `cache()`

```python
df.cache()
```

For DataFrames, the default storage level is typically **MEMORY_AND_DISK**.

Spark keeps the data in memory when possible and uses disk if it doesn't fit.

**Use when:** You want straightforward caching and don't need to specify a custom storage level.

---

### 2. `persist()`

```python
from pyspark import StorageLevel

df.persist(StorageLevel.MEMORY_AND_DISK)
```

`persist()` allows you to specify how the data should be stored.

Common options include:

```python
StorageLevel.MEMORY_ONLY
StorageLevel.MEMORY_AND_DISK
StorageLevel.DISK_ONLY
```

**Use when:** You need control over the storage strategy based on memory availability and workload requirements.

---

### Example

Suppose the same transformed DataFrame is used multiple times:

```python
df_clean = (
    df.filter(col("status") == "ACTIVE")
      .select("customer_id", "amount")
)

df_clean.cache()

df_clean.count()
df_clean.groupBy("customer_id").sum("amount")
df_clean.show()
```

Without caching, Spark may recompute the transformation lineage for each action.

With caching, Spark can reuse the computed DataFrame.

### `unpersist()`

Once the DataFrame is no longer needed:

```python
df_clean.unpersist()
```

This releases the cached data from executor storage.

### ⭐ Strong Interview Answer

> **“`cache()` and `persist()` both avoid repeated computation by storing intermediate data. `cache()` uses the default storage level, whereas `persist()` allows me to explicitly select a storage level such as `MEMORY_ONLY`, `MEMORY_AND_DISK`, or `DISK_ONLY`. I use caching when the default behavior is sufficient and persist when I need control over storage. I also remove the cached data using `unpersist()` when it is no longer required.”**

### Quick Comparison

| Feature       | `cache()`           | `persist()`                   |
| ------------- | ------------------- | ----------------------------- |
| Purpose       | Reuse computed data | Reuse computed data           |
| Storage level | Default             | User-defined                  |
| Flexibility   | Low                 | High                          |
| Example       | `df.cache()`        | `df.persist(MEMORY_AND_DISK)` |
| Remove cache  | `unpersist()`       | `unpersist()`                 |

**Interview tip:** Don't say "`cache()` means memory-only." For modern Spark DataFrames, the default caching behavior is generally **`MEMORY_AND_DISK`**, so always distinguish the API from the storage level.


## 5.

### Handling Schema Drift in ADF

> **“Schema drift occurs when the source structure changes unexpectedly—for example, a new column is added, a column is removed, or its data type changes. In ADF, I handle it using schema-drift-aware mapping, metadata-driven pipelines, validation, and controlled error handling.”**

### 1. Identify the Type of Schema Change

Typical cases:

* **New column added**
* Column removed
* Column renamed
* Data type changed
* Column order changed

---

### 2. Enable Schema Drift Where Appropriate

For **Mapping Data Flows**, I can enable **Allow schema drift** so that columns not explicitly defined in the projection can flow through the transformation.

I can also use **auto mapping** for suitable ingestion scenarios.

```text
Source
  ↓
Mapping Data Flow
  ↓
Allow Schema Drift
  ↓
Transformations
  ↓
Target
```

However, I don't blindly allow every schema change in critical tables because an unexpected data-type or business-column change can corrupt downstream processing.

---

### 3. Use Metadata-Driven Pipelines

For multiple tables/files, I maintain metadata such as:

| Table    | Source | Target | Load Type   | Expected Schema          |
| -------- | ------ | ------ | ----------- | ------------------------ |
| Customer | SQL    | Delta  | Incremental | customer_id, name, email |
| Orders   | SQL    | Delta  | Incremental | order_id, amount, date   |

ADF can read this metadata and dynamically construct the pipeline.

---

### 4. Validate Schema Before Loading

I compare the incoming schema with the expected schema.

For example:

```text
Expected:
customer_id INT
name        STRING
email       STRING

Incoming:
customer_id INT
name        STRING
email       STRING
phone       STRING
```

A new `phone` column can be handled according to the agreed business rule:

* Add it to the target
* Ignore it
* Store it for review
* Fail the pipeline

---

### 5. Handle Breaking Changes Separately

For a **non-breaking change**, such as adding an optional column:

```text
New Column → Validate → Accept → Continue
```

For a **breaking change**, such as:

```text
customer_id INT → customer_id STRING
```

I would generally **fail/quarantine the load**, alert the team, and investigate rather than silently converting the data.

---

### 6. Maintain Audit & Alerts

I capture:

* Source schema/version
* Pipeline run ID
* Changed columns
* Data type changes
* Record counts
* Failure reason

Then send alerts through the organization's monitoring/notification mechanism.

---

### ⭐ Strong Interview Answer

> **“I handle schema drift in ADF by first identifying whether the change is additive or breaking. For flexible ingestion pipelines, I use Mapping Data Flow's schema drift and auto-mapping capabilities. I maintain expected schema metadata and validate incoming schemas before loading. New non-breaking columns can be handled dynamically, while breaking changes such as data-type changes are quarantined or cause the pipeline to fail with an alert. I also maintain audit logs and schema versions so that downstream systems are protected and changes are traceable.”**

### Real-World Architecture

```text
Source
  ↓
ADF
  ↓
Schema Validation
  ↓
 ┌───────────────┐
 │ Schema Change?│
 └───────┬───────┘
         │
   ┌─────┴─────┐
   ↓           ↓
Non-Breaking  Breaking
   ↓           ↓
Accept      Quarantine
   ↓           ↓
Transform   Alert/Review
   ↓
ADLS / Delta / Target
```

**Key interview keywords:**
**Schema Drift → Mapping Data Flow → Allow Schema Drift → Auto Mapping → Metadata-Driven Pipeline → Schema Validation → Schema Versioning → Quarantine → Monitoring & Alerting**


## 6.

### Securely Managing Secrets & Credentials in Azure

> **“I never hardcode passwords, connection strings, API keys, or access tokens in ADF pipelines, Databricks notebooks, or source code. I use Azure Key Vault together with Managed Identity and RBAC to securely manage and access secrets.”**

### 1. Store Secrets in Azure Key Vault

I store sensitive information such as:

* Database passwords
* API keys
* Connection strings
* SAS tokens
* Service credentials

```text
ADF / Databricks
       ↓
 Azure Key Vault
       ↓
   Secret
```

---

### 2. Use Managed Identity

Instead of storing credentials for ADF itself, I enable **Managed Identity** for the Azure service.

For example:

```text
ADF Managed Identity
        ↓
   Azure Key Vault
        ↓
    Get Secret
```

The identity is authenticated by Azure, so I don't need to put a password or client secret inside the pipeline.

---

### 3. Configure RBAC / Access Policies

I give the Managed Identity only the permissions it requires.

For example:

```text
ADF Managed Identity
        ↓
Key Vault
        ↓
Secret Reader Access
```

This follows the **Principle of Least Privilege**.

---

### 4. Reference Key Vault from ADF

ADF Linked Services can use **Azure Key Vault** to retrieve secrets rather than storing credentials directly in the pipeline.

For example:

```text
ADF Linked Service
       ↓
Azure Key Vault
       ↓
DB Password
       ↓
Source Database
```

The actual password isn't exposed in the pipeline configuration.

---

### 5. Databricks Secret Management

For Databricks, I avoid putting secrets directly into notebooks.

I use **secret scopes / Azure-integrated secret management** and retrieve secrets at runtime.

Conceptually:

```python
password = dbutils.secrets.get(
    scope="prod-secrets",
    key="db-password"
)
```

The notebook contains the **secret reference**, not the actual password.

---

### 6. Secure Network Access

For sensitive workloads, I also use:

* Private Endpoints
* VNet integration
* Network security controls
* Firewall rules
* Restricted public access

This protects the communication path in addition to protecting the credentials themselves.

---

### 7. Rotation & Monitoring

I implement:

* Secret rotation
* Expiration policies where applicable
* Key Vault logging
* Azure Monitor alerts
* Access auditing

If a credential is compromised, it can be rotated without modifying every pipeline that consumes it.

### ⭐ Strong Interview Answer

> **“In Azure, I use Azure Key Vault as the central secret store and avoid hardcoding credentials in ADF, Databricks, or code repositories. I use Managed Identity for services such as ADF to authenticate to Key Vault without storing another credential. Access is controlled through RBAC and least-privilege permissions. ADF Linked Services can reference Key Vault secrets, while Databricks can retrieve secrets securely at runtime. I also use private endpoints and network controls for sensitive workloads, along with secret rotation, monitoring, and auditing.”**

### Simple Architecture

```text
                    Azure Key Vault
                   /              \
                  /                \
        Managed Identity       Secret Scopes
               ↓                     ↓
             ADF               Databricks
               ↓                     ↓
        Source / Target       ADLS / Database
```

**Key interview keywords:**
**Azure Key Vault → Managed Identity → RBAC → Least Privilege → Secret Rotation → ADF Linked Service → Databricks Secret Scope → Private Endpoint → Monitoring & Auditing**


## 🔴 Round 3 – Managerial / Behavioral

1. Tell me about a project where you handled **large-scale data processing**.
2. How do you deal with **tight deadlines in client projects**?
3. Describe a situation where you **improved pipeline performance**.
4. How do you communicate with **non-technical stakeholders**?
5. What will you do if your **data pipeline fails in production**?
